# Lecture 28 Worksheet: Many-to-Many RNNs and Encoder-Decoder
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`. Tasks 1 to 3 and 5 use numpy only. Tasks 4 and 6 train small networks with TensorFlow, which Colab already has (about a minute each on a CPU).

## Task 1: Which type of RNN?
Use the three questions from slide 3. For each task, fill in one of: `"many-to-one"`, `"same length"`, `"same length, bidirectional"`, `"length changes"`.

In [1]:
types = {
    "Is this review positive?":                              "many-to-one",
    "Mark every name and date in a sentence":               "same length",
    "Mark each word, where the word AFTER it matters too":  "same length, bidirectional",
    "Turn a long email into a 2-line summary":              "length changes",
}
answers = ["many-to-one", "same length", "same length, bidirectional", "length changes"]
for q, a in types.items(): print(f"{a:28s} <- {q}")
assert list(types.values()) == answers; print("OK")

many-to-one                  <- Is this review positive?
same length                  <- Mark every name and date in a sentence
same length, bidirectional   <- Mark each word, where the word AFTER it matters too
length changes               <- Turn a long email into a 2-line summary
OK


## Task 2: An output at every step
Same cell as Lecture 27. This time keep the output at every step: `y = W_hy * h + b_y`. Compare with slide 6 (1.42, 2.16, 2.40).

In [2]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
W_xh, W_hh, b_h = 0.5, 0.4, 0.0
W_hy, b_y = 2.0, 0.5

def rnn_outputs(x):
    h, hs, ys = 0.0, [], []
    for x_t in x:
        h = np.tanh(W_hh * h + W_xh * x_t + b_h)
        hs.append(h)
        ys.append(W_hy * h + b_y)      # read the memory at EVERY step
    return np.array(hs), np.array(ys)

hs, ys = rnn_outputs([1, 2, 3])
print("memories:", hs, " outputs:", ys)
assert np.allclose(ys, [1.42, 2.16, 2.40], atol=0.01); print("OK")

memories: [0.462 0.829 0.95 ]  outputs: [1.424 2.158 2.4  ]
OK


## Task 3: Tagging with an RNN, against a model that reads x only
Toy task (ours): for each of 20 numbers, the label is 1 if the **running total so far** is positive. The "x only" model sees one number at a time, so it has no memory.

In [3]:
import tensorflow as tf
L = tf.keras.layers
T = 20

def make_cum(N, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    y = (np.cumsum(x[:, :, 0], axis=1) > 0).astype("float32")
    return x, y

xtr, ytr = make_cum(3000, 0); xte, yte = make_cum(1000, 1)

def fit_eval(layers, xtr, ytr, xte, yte, seed=0):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential(layers)
    m.compile(tf.keras.optimizers.Adam(0.01), "binary_crossentropy")
    m.fit(xtr, ytr, epochs=15, batch_size=64, verbose=0)
    p = (m.predict(xte, verbose=0)[..., 0] > 0.5)
    return p, m

p0, _ = fit_eval([L.Input((T, 1)), L.Dense(16, activation="relu"), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
p1, m1 = fit_eval([L.Input((T, 1)), L.SimpleRNN(16, return_sequences=True), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
acc_x_only = (p0 == yte).mean(); acc_rnn = (p1 == yte).mean()
print("x only:", round(acc_x_only, 3), " RNN:", round(acc_rnn, 3), " output shape:", m1.output_shape)
assert m1.output_shape == (None, 20, 1) and acc_rnn > acc_x_only + 0.1; print("OK")

x only: 0.636  RNN: 0.982  output shape: (None, 20, 1)
OK


## Task 4: Does the label need the future? Forward vs bidirectional
New label: `1` if the **next** number is positive. The last position has no next number, so we do not score it.

In [4]:
def make_next(N, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    y = np.zeros((N, T), "float32"); y[:, :-1] = (x[:, 1:, 0] > 0)
    return x, y

xtr, ytr = make_next(3000, 0); xte, yte = make_next(1000, 1)

pf, _ = fit_eval([L.Input((T, 1)), L.SimpleRNN(16, return_sequences=True), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
pb, _ = fit_eval([L.Input((T, 1)), L.Bidirectional(L.SimpleRNN(16, return_sequences=True)), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
acc_fwd = (pf[:, :-1] == yte[:, :-1]).mean(); acc_bi = (pb[:, :-1] == yte[:, :-1]).mean()
print("forward RNN:", round(acc_fwd, 3), " bidirectional:", round(acc_bi, 3))
assert acc_fwd < 0.6 and acc_bi > 0.9; print("OK")

forward RNN: 0.5  bidirectional: 0.998
OK


**Question (2 sentences):** when can you NOT use a bidirectional RNN, even if it would be more accurate?

## Task 5: The decoder by hand, in code
Encoder = the Task 2 cell on 1, 2, 3, so the context is `c = h3 = 0.950`. Decoder rule (slide 14): `s = tanh(0.5 * s_prev + 0.3 * y_prev)`, `y = 2 * s + 0.5`, start `s = c`, `y_prev = 0`. Match 1.384, 1.625, 1.792.

In [5]:
c = hs[-1]
s, y_prev, outs = c, 0.0, []
for step in range(3):
    s = np.tanh(0.5 * s + 0.3 * y_prev)
    y = 2 * s + 0.5
    outs.append(y)
    y_prev = y            # feed the output back as the next input
print("decoder outputs:", np.array(outs))
assert np.allclose(outs, [1.384, 1.625, 1.792], atol=0.002); print("OK")

decoder outputs: [1.384 1.625 1.792]
OK


## Task 6: A small encoder-decoder that reverses 5 digits
Input `3 1 4 1 5` should give `5 1 4 1 3`. The output has the same length here, but the first answer needs the last input, so we read everything first. Digits are one-hot (10 values); the decoder input has an extra symbol (id 10) for `<start>`.

In [6]:
T6 = 5
def onehot(a, n): return np.eye(n, dtype="float32")[a]

r = np.random.RandomState(0)
X = r.randint(0, 10, (4000, T6)); Y = X[:, ::-1]
Xte = r.randint(0, 10, (1000, T6)); Yte = Xte[:, ::-1]
# teacher forcing: decoder input = <start> + true answer shifted by one
dec_in = np.concatenate([np.full((len(X), 1), 10), Y[:, :-1]], axis=1)

tf.keras.utils.set_random_seed(0)
enc_in = L.Input((T6, 10)); dec_inp = L.Input((T6, 11))
enc = L.SimpleRNN(64, return_state=True)
dec = L.SimpleRNN(64, return_sequences=True, return_state=True)
dense = L.Dense(10)
state = enc(enc_in)[1:]                       # the context vector
seq = dec(dec_inp, initial_state=state)[0]
model = tf.keras.Model([enc_in, dec_inp], dense(seq))
print("output shape:", model.output_shape)
model.compile(tf.keras.optimizers.Adam(0.003), tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model.fit([onehot(X, 10), onehot(dec_in, 11)], Y, epochs=30, batch_size=64, verbose=0)

# inference: the decoder uses its OWN previous output
st = list(enc(onehot(Xte, 10))[1:])
tok = np.full((len(Xte),), 10); preds = []
for t in range(T6):
    o = dec(onehot(tok, 11)[:, None, :], initial_state=st)
    st = list(o[1:])
    tok = dense(o[0])[:, 0, :].numpy().argmax(1)
    preds.append(tok)
P = np.stack(preds, 1)
exact = (P == Yte).all(axis=1).mean()
print("example:", Xte[0], "->", P[0], " exact match:", round(exact, 3))
assert model.output_shape == (None, 5, 10) and exact > 0.9; print("OK")

output shape: (None, 5, 10)
example: [9 3 9 3 2] -> [2 3 9 3 9]  exact match: 1.0
OK


**Question (2 sentences):** the model sees the true answers during training but not at test time. What goes wrong when you try a length of 20, and why does one context vector explain it?

## Task 7: Your own project
In 3 to 4 sentences: which of the five RNN types does your project need (one-to-one, one-to-many, many-to-one, same-length many-to-many, length-changing many-to-many)? Do you need bidirectional? What simple baseline would you try first?

*Your answer:*